In [68]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [69]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [70]:
# query_com = """
# SELECT DISTINCT Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2026-06-20'
#     AND v.Fecha <= '2026-06-20'
#     AND Ubicacion = 'Social Selling'
#     AND v.Venta_Neta > 0
# GROUP BY  Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_com, engine)

# df_ventas.head()

In [71]:
# CONSULTAS CUMPLEAÑOS

query_data = """
SELECT DISTINCT Pedido,
    v.Cliente, 
	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
	cc.Email, cc.Celular, 
	v.Fecha AS Fecha_Venta,
	Canal,
    SUM(Venta_Neta) AS Venta 
FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'JUNIO_2026')
	AND Fecha BETWEEN '2026-06-30' AND '2026-06-30'
    AND Ubicacion IN ('Social Selling') 
GROUP BY  Pedido,
    v.Cliente,
    cc.Nombres + ' ' +  cc.Apellidos,
    cc.Email,
    cc.Celular,
    v.Fecha,
    v.Canal
"""


# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_data, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,100935,C52008840,LINA MARIA GARCIA MORA,linamaga@hotmail.com,3124013522,2026-06-30,CHATCENTER WEB,95163.98
1,100936,C63543937,HELGA HERNANDEZ,helgahdez@gmail.com,3003918424,2026-06-30,CHATCENTER WEB,225620.55
2,100937,C60309333,PAULINA SUÁREZ SUÁREZ. 2 PARES XL BEIGE,pauli_cambios@hotmail.com,3158760291,2026-06-30,CHATCENTER WEB,159446.93
3,100938,C51857251,ELSA PATRICIA MANRIQUE OSPINA,epmo14@gmail.com,3157864558,2026-06-30,CHATCENTER WEB,138961.40
4,100942,C1103097771,VANESSA MONTERROZA,vanessamonterroza@hotmail.com,3022428241,2026-06-30,CHATCENTER WEB,234443.70


In [72]:
# Leer archivo
df_social = pd.read_excel("Atribucion.xlsx")

df_social['telefono'] = df_social['telefono'].astype(str).str[2:]

In [73]:
# Normalizar df_ventas
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por celular
df_atribucion = pd.merge(df_ventas, df_social, left_on='Celular', right_on='telefono' , how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.2f}")
# Resultado final
df_atribucion.head()

Total de Ventas: 24
Total de venta atribuida: 6,782,701.11


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,telefono
0,100935,C52008840,LINA MARIA GARCIA MORA,linamaga@hotmail.com,3124013522,2026-06-30,CHATCENTER WEB,95163.98,3124013522
1,100936,C63543937,HELGA HERNANDEZ,helgahdez@gmail.com,3003918424,2026-06-30,CHATCENTER WEB,225620.55,3003918424
2,100937,C60309333,PAULINA SUÁREZ SUÁREZ. 2 PARES XL BEIGE,pauli_cambios@hotmail.com,3158760291,2026-06-30,CHATCENTER WEB,159446.93,3158760291
3,100942,C1103097771,VANESSA MONTERROZA,vanessamonterroza@hotmail.com,3022428241,2026-06-30,CHATCENTER WEB,234443.70,3022428241
4,100944,C43627185,CATALINA MESA,mesacata@hotmail.com,3104047797,2026-06-30,CHATCENTER WEB,151254.00,3104047797


In [74]:
df_atribucion.to_excel('Ventas.xlsx', index=False)